In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


def _find_project_root(start: Path) -> Path:
    """
    Locate the repository root by searching upward for processed outputs.
    """
    for candidate in [start, *start.parents]:
        processed_dir = candidate / "data" / "processed"
        reports_dir = candidate / "reports" / "tables"

        if processed_dir.exists() and reports_dir.exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. Expected data/processed and reports/tables directories."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())

US_IV_PATH = PROJECT_ROOT / "data" / "processed" / "us_iv.parquet"
INDIA_IV_PATH = PROJECT_ROOT / "data" / "processed" / "india_iv.parquet"

US_VRP_PATH = PROJECT_ROOT / "data" / "processed" / "us_vrp.parquet"
INDIA_VRP_PATH = PROJECT_ROOT / "data" / "processed" / "india_vrp.parquet"

VRP_SUMMARY_PATH = PROJECT_ROOT / "reports" / "tables" / "vrp_summary.csv"
VRP_METADATA_PATH = PROJECT_ROOT / "reports" / "tables" / "vrp_metadata.json"
CALENDAR_MISMATCH_PATH = PROJECT_ROOT / "reports" / "tables" / "calendar_mismatches.csv"

required_paths = {
    "US_IV_PATH": US_IV_PATH,
    "INDIA_IV_PATH": INDIA_IV_PATH,
    "US_VRP_PATH": US_VRP_PATH,
    "INDIA_VRP_PATH": INDIA_VRP_PATH,
    "VRP_SUMMARY_PATH": VRP_SUMMARY_PATH,
    "VRP_METADATA_PATH": VRP_METADATA_PATH,
    "CALENDAR_MISMATCH_PATH": CALENDAR_MISMATCH_PATH,
}

missing_paths = [name for name, path in required_paths.items() if not path.exists()]
if missing_paths:
    print(f"Project root: {PROJECT_ROOT}")
    print("Missing required notebook inputs:")
    for name in missing_paths:
        print(f"  - {name}: {required_paths[name]}")
    raise FileNotFoundError(
        "One or more processed outputs or reports are missing. Run the feature build pipeline first."
    )

us_iv = pd.read_parquet(US_IV_PATH)
india_iv = pd.read_parquet(INDIA_IV_PATH)

us_vrp = pd.read_parquet(US_VRP_PATH)
india_vrp = pd.read_parquet(INDIA_VRP_PATH)

for df in [us_iv, india_iv, us_vrp, india_vrp]:
    df["date"] = pd.to_datetime(df["date"])

print("US IV:", us_iv.shape)
print("India IV:", india_iv.shape)
print("US VRP:", us_vrp.shape)
print("India VRP:", india_vrp.shape)

display(us_vrp.head())
display(india_vrp.head())


In [ ]:
PRIMARY_COLUMNS = [
    "date",
    "market",
    "underlying_symbol",
    "iv_symbol",
    "iv_close",
    "iv_ann",
    "rv_gk_daily",
    "rv_gk_22d_ann",
    "rv_gk_22d_ann_lag1",
    "vrp_backward_gk",
    "vrp_backward_gk_positive",
    "rv_gk_22d_forward_ann_label",
    "vrp_forward_expost_gk_label",
    "feature_allowed",
]

ROBUSTNESS_COLUMNS = [
    "rv_cc_22d_ann_lag1",
    "vrp_backward_cc",
    "vrp_backward_cc_positive",
    "rv_parkinson_22d_ann_lag1",
    "vrp_backward_parkinson",
    "vrp_backward_parkinson_positive",
    "rv_rs_22d_ann_lag1",
    "vrp_backward_rs",
    "vrp_backward_rs_positive",
    "rv_yz_22d_ann_lag1",
    "vrp_backward_yz",
    "vrp_backward_yz_positive",
]

FORBIDDEN_COLUMNS = [
    "rv_yz_daily",
    "vrp_forward_expost_cc_label",
    "vrp_forward_expost_parkinson_label",
    "vrp_forward_expost_rs_label",
    "vrp_forward_expost_yz_label",
]

def check_vrp_panel(df: pd.DataFrame, name: str) -> None:
    missing_primary = [col for col in PRIMARY_COLUMNS if col not in df.columns]
    missing_robustness = [col for col in ROBUSTNESS_COLUMNS if col not in df.columns]
    forbidden_present = [col for col in FORBIDDEN_COLUMNS if col in df.columns]

    print(name)
    print("Missing primary:", missing_primary)
    print("Missing robustness:", missing_robustness)
    print("Forbidden present:", forbidden_present)
    print("First feature_allowed index:", df.index[df["feature_allowed"] == True].min())

    assert not missing_primary
    assert not missing_robustness
    assert not forbidden_present

check_vrp_panel(us_vrp, "US")
check_vrp_panel(india_vrp, "INDIA")

In [ ]:
FEATURE_COLUMNS = [
    "iv_ann",
    "rv_gk_22d_ann_lag1",
    "vrp_backward_gk",
    "vrp_backward_gk_positive",
]

LABEL_COLUMNS = [
    "rv_gk_22d_forward_ann_label",
    "vrp_forward_expost_gk_label",
]

FORBIDDEN_SUBSTRINGS = ["future", "forward", "expost", "label"]

for col in FEATURE_COLUMNS:
    for token in FORBIDDEN_SUBSTRINGS:
        assert token not in col.lower(), f"Forbidden token {token} found in feature column {col}"

for col in LABEL_COLUMNS:
    assert col not in FEATURE_COLUMNS

print("Feature/label separation OK")

In [ ]:
AUDIT_COLUMNS = PRIMARY_COLUMNS + ROBUSTNESS_COLUMNS

def missing_audit(df: pd.DataFrame, market: str) -> pd.DataFrame:
    rows = []

    for col in AUDIT_COLUMNS:
        if col not in df.columns:
            continue

        rows.append(
            {
                "market": market,
                "column": col,
                "missing": int(df[col].isna().sum()),
                "count": int(df[col].notna().sum()),
                "first_valid_index": df[col].first_valid_index(),
            }
        )

    return pd.DataFrame(rows)

missing = pd.concat(
    [
        missing_audit(us_vrp, "US"),
        missing_audit(india_vrp, "INDIA"),
    ],
    ignore_index=True,
)

display(missing)

In [ ]:
vrp_summary = pd.read_csv(VRP_SUMMARY_PATH)
calendar_mismatches = pd.read_csv(CALENDAR_MISMATCH_PATH)

display(vrp_summary)
display(calendar_mismatches)

In [ ]:
def plot_iv_vs_rv(df: pd.DataFrame, market: str) -> None:
    fig, ax = plt.subplots(figsize=(14, 5))

    ax.plot(df["date"], df["iv_ann"], label="iv_ann")
    ax.plot(df["date"], df["rv_gk_22d_ann_lag1"], label="rv_gk_22d_ann_lag1")

    ax.set_title(f"{market}: IV vs lagged realised variance")
    ax.set_xlabel("Date")
    ax.set_ylabel("Annualised variance")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.show()

plot_iv_vs_rv(us_vrp, "US")
plot_iv_vs_rv(india_vrp, "India")

In [ ]:
def plot_backward_vrp(df: pd.DataFrame, market: str) -> None:
    fig, ax = plt.subplots(figsize=(14, 5))

    ax.plot(df["date"], df["vrp_backward_gk"], label="vrp_backward_gk")
    ax.axhline(0.0, linewidth=1)

    ax.set_title(f"{market}: backward VRP")
    ax.set_xlabel("Date")
    ax.set_ylabel("Variance spread")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.show()

plot_backward_vrp(us_vrp, "US")
plot_backward_vrp(india_vrp, "India")

In [ ]:
def plot_forward_expost_label(df: pd.DataFrame, market: str) -> None:
    fig, ax = plt.subplots(figsize=(14, 5))

    ax.plot(
        df["date"],
        df["vrp_forward_expost_gk_label"],
        label="vrp_forward_expost_gk_label",
    )
    ax.axhline(0.0, linewidth=1)

    ax.set_title(f"{market}: forward ex-post VRP label")
    ax.set_xlabel("Date")
    ax.set_ylabel("Variance spread")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.show()

plot_forward_expost_label(us_vrp, "US")
plot_forward_expost_label(india_vrp, "India")

In [ ]:
ROBUSTNESS_VRP_COLUMNS = [
    "vrp_backward_gk",
    "vrp_backward_cc",
    "vrp_backward_parkinson",
    "vrp_backward_rs",
    "vrp_backward_yz",
]

def plot_robustness_vrp(df: pd.DataFrame, market: str) -> None:
    fig, ax = plt.subplots(figsize=(14, 6))

    for col in ROBUSTNESS_VRP_COLUMNS:
        if col in df.columns:
            ax.plot(df["date"], df[col], label=col)

    ax.axhline(0.0, linewidth=1)
    ax.set_title(f"{market}: backward VRP robustness comparison")
    ax.set_xlabel("Date")
    ax.set_ylabel("Variance spread")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.show()

plot_robustness_vrp(us_vrp, "US")
plot_robustness_vrp(india_vrp, "India")

In [ ]:
COMPARE_COLUMNS = [
    "iv_ann",
    "rv_gk_22d_ann_lag1",
    "vrp_backward_gk",
    "vrp_forward_expost_gk_label",
]

ROBUSTNESS_COMPARE_COLUMNS = [
    "vrp_backward_cc",
    "vrp_backward_parkinson",
    "vrp_backward_rs",
    "vrp_backward_yz",
]

comparison = []

for market, df in [("US", us_vrp), ("INDIA", india_vrp)]:
    for col in COMPARE_COLUMNS + ROBUSTNESS_COMPARE_COLUMNS:
        if col not in df.columns:
            continue

        values = pd.to_numeric(df[col], errors="coerce")

        comparison.append(
            {
                "market": market,
                "column": col,
                "mean": values.mean(),
                "median": values.median(),
                "std": values.std(),
                "p05": values.quantile(0.05),
                "p95": values.quantile(0.95),
                "positive_ratio": (values.dropna() > 0).mean(),
            }
        )

comparison_df = pd.DataFrame(comparison)
display(comparison_df)
